In [1]:
import pandas as pd

Формулировка задачи:
На основе предоставленных данных необходимо получить итоговый отчет, который содержит следующую информацию для каждого активного клиента за 2024 год:

	Общую сумму транзакций по типам операций.

	Среднюю сумму транзакций по типам операций.

	Количество транзакций по типам операций.

	Дату последней транзакции.

In [4]:
data_transactions = {
    'transaction_id': [1, 2, 3, 4, 5, 6, 7, 8],
    'client_id': [101, 102, 101, 103, 102, 101, 103, 102],
    'transaction_date': ['2024-01-15', '2024-02-20', '2024-03-10', '2024-04-05', '2024-05-12', '2023-12-25', '2024-06-01', '2024-07-18'],
    'amount': ['1000.50', '200.75', '150.00', '500.25', '300.50', '1200.00', '700.00', '400.00'],
    'currency': ['USD', 'EUR', 'USD', 'USD', 'EUR', 'USD', 'USD', 'EUR'],
    'transaction_type': ['deposit', 'withdrawal', 'deposit', 'withdrawal', 'deposit', 'deposit', 'withdrawal', 'withdrawal']
}

data_clients = {
    'client_id': [101, 102, 103],
    'client_name': ['Иван Иванов', 'Петр Петров', 'Сидор Сидоров'],
    'registration_date': ['2023-12-01', '2024-01-10', '2024-02-15'],
    'client_status': ['active', 'active', 'inactive']
}

transactions = pd.DataFrame(data_transactions)
clients = pd.DataFrame(data_clients)

In [5]:
# даты
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"])
clients["registration_date"] = pd.to_datetime(clients["registration_date"])

In [8]:
# суммы
transactions["amount"] = transactions["amount"].astype(float)

In [9]:
transactions

,transaction_id,client_id,transaction_date,amount,currency,transaction_type
0,1,101,2024-01-15,1000.50,USD,deposit
1,2,102,2024-02-20,200.75,EUR,withdrawal
2,3,101,2024-03-10,150.00,USD,deposit
3,4,103,2024-04-05,500.25,USD,withdrawal
4,5,102,2024-05-12,300.50,EUR,deposit
5,6,101,2023-12-25,1200.00,USD,deposit
6,7,103,2024-06-01,700.00,USD,withdrawal
7,8,102,2024-07-18,400.00,EUR,withdrawal


In [10]:
clients

,client_id,client_name,registration_date,client_status
0,101,Иван Иванов,2023-12-01,active
1,102,Петр Петров,2024-01-10,active
2,103,Сидор Сидоров,2024-02-15,inactive


In [11]:
# только активные клиенты
active_clients = clients[clients["client_status"] == "active"]

In [12]:
# только операции 2024 года
transactions_2024 = transactions[
    transactions["transaction_date"].dt.year == 2024
]

In [13]:
df = transactions_2024.merge(active_clients, on="client_id")
df

,transaction_id,client_id,transaction_date,amount,currency,transaction_type,client_name,registration_date,client_status
0,1,101,2024-01-15,1000.50,USD,deposit,Иван Иванов,2023-12-01,active
1,2,102,2024-02-20,200.75,EUR,withdrawal,Петр Петров,2024-01-10,active
2,3,101,2024-03-10,150.00,USD,deposit,Иван Иванов,2023-12-01,active
3,5,102,2024-05-12,300.50,EUR,deposit,Петр Петров,2024-01-10,active
4,8,102,2024-07-18,400.00,EUR,withdrawal,Петр Петров,2024-01-10,active


In [14]:
# группировка
report = df.groupby(
    ["client_id", "client_name", "transaction_type"]
).agg(
    total_amount=("amount", "sum"),
    average_amount=("amount", "mean"),
    transaction_count=("amount", "count"),
    last_transaction_date=("transaction_date", "max")
).reset_index()

In [15]:
# сортировка
report = report.sort_values(
    by="total_amount",
    ascending=False
)

report

,client_id,client_name,transaction_type,total_amount,average_amount,transaction_count,last_transaction_date
0,101,Иван Иванов,deposit,1150.50,575.250,2,2024-03-10
2,102,Петр Петров,withdrawal,600.75,300.375,2,2024-07-18
1,102,Петр Петров,deposit,300.50,300.500,1,2024-05-12
